<a href="https://colab.research.google.com/github/ananya-6653/Supermarket_MiniProject/blob/main/Billing%26InventoryAnalyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

# Set seed so your random data stays reproducible during your viva
rng = np.random.default_rng(seed=42)
NUM_ROWS = 100_000  # 1 Lakh records

# Product catalog: (Item_ID, Product_Name, Category, Unit_Price_INR, Cost_Price_INR)
catalog = [
    ("ITM101", "Amul Taaza Milk 1L", "Dairy", 74, 65),
    ("ITM102", "Amul Butter 100g", "Dairy", 58, 50),
    ("ITM103", "Mother Dairy Paneer 200g", "Dairy", 95, 78),
    ("ITM104", "Britannia Curd 400g", "Dairy", 42, 34),
    ("ITM105", "Aashirvaad Atta 5kg", "Staples", 265, 225),
    ("ITM106", "India Gate Basmati Rice 1kg", "Staples", 145, 118),
    ("ITM107", "Tata Sampann Toor Dal 1kg", "Staples", 185, 152),
    ("ITM108", "Fortune Sunlite Oil 1L", "Staples", 138, 119),
    ("ITM109", "Tata Salt 1kg", "Staples", 28, 22),
    ("ITM110", "Parle-G Gold 1kg", "Snacks", 130, 102),
    ("ITM111", "Lay's India's Magic Masala", "Snacks", 20, 15),
    ("ITM112", "Haldiram's Aloo Bhujia 400g", "Snacks", 110, 86),
    ("ITM113", "Maggi 2-Minute Noodles 4-Pack", "Snacks", 56, 46),
    ("ITM114", "Tata Tea Gold 500g", "Beverages", 280, 232),
    ("ITM115", "Nescafe Classic Jar 50g", "Beverages", 175, 142),
    ("ITM116", "Real Mixed Fruit Juice 1L", "Beverages", 125, 96),
    ("ITM117", "Coca-Cola 750ml", "Beverages", 45, 35),
    ("ITM118", "Dettol Liquid Handwash 200ml", "Personal Care", 99, 76),
    ("ITM119", "Colgate Strong Teeth 200g", "Personal Care", 115, 88),
    ("ITM120", "Dove Cream Beauty Bar 3-Pack", "Personal Care", 165, 128),
]

# Assign current stock levels per unique item (some intentionally below 10 for alerts)
current_stock_map = {
    item[0]: int(rng.integers(3, 85)) for item in catalog
}

# Randomly pick 1 lakh product entries
indices = rng.integers(0, len(catalog), size=NUM_ROWS)

item_ids = [catalog[i][0] for i in indices]
product_names = [catalog[i][1] for i in indices]
categories = [catalog[i][2] for i in indices]
unit_prices = np.array([catalog[i][3] for i in indices], dtype=np.float64)
cost_prices = np.array([catalog[i][4] for i in indices], dtype=np.float64)

# Simulate ~25,000 customer bills (average 4 items per basket)
transaction_ids = [f"TXN{num}" for num in rng.integers(100001, 125001, size=NUM_ROWS)]
quantities_sold = rng.integers(1, 12, size=NUM_ROWS)
stock_remaining = [current_stock_map[iid] for iid in item_ids]

# Build DataFrame and save to CSV
raw_df = pd.DataFrame({
    "Transaction_ID": transaction_ids,
    "Item_ID": item_ids,
    "Product_Name": product_names,
    "Category": categories,
    "Unit_Price": unit_prices,
    "Cost_Price": cost_prices,
    "Quantity_Sold": quantities_sold,
    "Stock_Remaining": stock_remaining
})

raw_df.to_csv("kirana_sales_1lakh.csv", index=False)
print(f"Generated {len(raw_df):,} rows successfully and saved to 'kirana_sales_1lakh.csv'!")
raw_df.head()

Generated 100,000 rows successfully and saved to 'kirana_sales_1lakh.csv'!


,Transaction_ID,Item_ID,Product_Name,Category,Unit_Price,Cost_Price,Quantity_Sold,Stock_Remaining
0,TXN114180,ITM111,Lay's India's Magic Masala,Snacks,20.0,15.0,3,46
1,TXN119531,ITM108,Fortune Sunlite Oil 1L,Staples,138.0,119.0,11,60
2,TXN102556,ITM104,Britannia Curd 400g,Dairy,42.0,34.0,7,38
3,TXN123604,ITM119,Colgate Strong Teeth 200g,Personal Care,115.0,88.0,5,71
4,TXN111922,ITM116,Real Mixed Fruit Juice 1L,Beverages,125.0,96.0,5,67


In [2]:
# Load the dataset
df = pd.read_csv("kirana_sales_1lakh.csv")

# Extract columns as NumPy arrays for high-speed vectorized math
unit_price_arr = df["Unit_Price"].to_numpy()
cost_price_arr = df["Cost_Price"].to_numpy()
qty_arr = df["Quantity_Sold"].to_numpy()

# 1. Gross Amount & Total Cost per line item
gross_amount = unit_price_arr * qty_arr
total_cost = cost_price_arr * qty_arr

# 2. Bulk Discount via np.where(): 10% off if line item total exceeds Rs. 1,000
discount_amount = np.where(gross_amount > 1000, gross_amount * 0.10, 0.0)
net_revenue = gross_amount - discount_amount

# 3. Profit & Profit Margin (%) per line item
net_profit = net_revenue - total_cost
profit_margin_pct = np.round((net_profit / net_revenue) * 100, 2)

# Attach computed arrays back to the Pandas DataFrame
df["Gross_Amount"] = gross_amount
df["Discount_Applied"] = discount_amount
df["Net_Revenue"] = net_revenue
df["Net_Profit"] = net_profit
df["Profit_Margin_%"] = profit_margin_pct

# 4. Calculate Average Basket Value (total bill per unique Transaction_ID)
basket_totals = df.groupby("Transaction_ID")["Net_Revenue"].sum().to_numpy()
avg_basket_value = np.mean(basket_totals)
max_basket_value = np.max(basket_totals)

print("--- Store-Wide KPIs (1,00,000 Line Items) ---")
print(f"Total Net Revenue     : ₹{np.sum(net_revenue):,.2f}")
print(f"Total Bulk Discounts  : ₹{np.sum(discount_amount):,.2f}")
print(f"Total Net Profit      : ₹{np.sum(net_profit):,.2f}")
print(f"Overall Profit Margin : {np.mean(profit_margin_pct):.2f}%")
print(f"Average Basket Value  : ₹{avg_basket_value:,.2f} (Across {len(basket_totals):,} bills)")
print(f"Highest Single Bill   : ₹{max_basket_value:,.2f}")

--- Store-Wide KPIs (1,00,000 Line Items) ---
Total Net Revenue     : ₹66,543,180.40
Total Bulk Discounts  : ₹3,969,505.60
Total Net Profit      : ₹9,257,526.40
Overall Profit Margin : 17.01%
Average Basket Value  : ₹2,712.51 (Across 24,532 bills)
Highest Single Bill   : ₹13,547.00


In [4]:
def get_category_performance(data: pd.DataFrame) -> pd.DataFrame:
    """Groups sales by Category to find highest-earning sections."""
    summary = (
        data.groupby("Category", as_index=False)
        .agg(
            Total_Units_Sold=("Quantity_Sold", "sum"),
            Gross_Revenue=("Gross_Amount", "sum"),
            Discounts_Given=("Discount_Applied", "sum"),
            Net_Revenue=("Net_Revenue", "sum"),
            Total_Profit=("Net_Profit", "sum"),
            Avg_Margin_Pct=("Profit_Margin_%", "mean"),
        )
        .sort_values(by="Net_Revenue", ascending=False)
    )
    summary["Avg_Margin_Pct"] = summary["Avg_Margin_Pct"].round(2)
    return summary


def get_product_revenue_ranking(data: pd.DataFrame) -> pd.DataFrame:
    """Calculates total revenue and units sold per product."""
    prod_summary = (
        data.groupby(["Item_ID", "Product_Name", "Category"], as_index=False)
        .agg(
            Units_Sold=("Quantity_Sold", "sum"),
            Net_Revenue=("Net_Revenue", "sum"),
            Net_Profit=("Net_Profit", "sum"),
        )
        .sort_values(by="Net_Revenue", ascending=False)
    )
    return prod_summary


def get_low_stock_alerts(data: pd.DataFrame, threshold: int = 10) -> pd.DataFrame:
    """Filters unique items where Stock_Remaining is below the threshold."""
    unique_inventory = data[
        ["Item_ID", "Product_Name", "Category", "Unit_Price", "Stock_Remaining"]
    ].drop_duplicates(subset=["Item_ID"])

    low_stock = unique_inventory[
        unique_inventory["Stock_Remaining"] < threshold
    ].sort_values(by="Stock_Remaining", ascending=True)
    return low_stock


def export_all_reports(data: pd.DataFrame) -> None:
    """Exports processed summaries and low-stock alerts to CSV files."""
    cat_df = get_category_performance(data)
    prod_df = get_product_revenue_ranking(data)
    low_stock_df = get_low_stock_alerts(data, threshold=10)

    cat_df.to_csv("report_category_sales.csv", index=False)
    prod_df.to_csv("report_product_revenue.csv", index=False)
    low_stock_df.to_csv("report_low_stock_alerts.csv", index=False)
    print("\nExported 3 files to Colab storage:")
    print("1. report_category_sales.csv")
    print("2. report_product_revenue.csv")
    print("3. report_low_stock_alerts.csv")

In [7]:
def run_kirana_analyzer():
    while True:
        print("\n" + "=" * 52)
        print("  KIRANA STORE BILLING & INVENTORY ANALYZER (1L ROWS)")
        print("=" * 52)
        print("1. View Category-Wise Revenue & Profit Breakdown")
        print("2. View Top 10 Highest-Earning Products")
        print("3. Check Low-Stock Inventory Alerts (Stock < 10)")
        print("4. View Bulk Discount & Basket Value Statistics")
        print("5. Export All Processed Reports to CSV")
        print("6. Exit")

        choice = input("\nEnter your choice (1-6): ").strip()

        if choice == "1":
            print("\n--- Category Performance ---")
            print(get_category_performance(df).to_string(index=False))

        elif choice == "2":
            print("\n--- Top 10 Products by Net Revenue ---")
            print(get_product_revenue_ranking(df).head(10).to_string(index=False))

        elif choice == "3":
            alerts = get_low_stock_alerts(df, threshold=10)
            print(f"\n--- Low-Stock Alerts ({len(alerts)} Items Below 10 Units) ---")
            print(alerts.to_string(index=False))

        elif choice == "4":
            discounted_rows = np.sum(df["Discount_Applied"] > 0)
            baskets = df.groupby("Transaction_ID")["Net_Revenue"].sum().to_numpy()
            print("\n--- Discount & Basket Analytics ---")
            print(f"Line items eligible for 10% bulk discount : {discounted_rows:,} / {len(df):,}")
            print(f"Total discount amount given               : ₹{df['Discount_Applied'].sum():,.2f}")
            print(f"Average customer basket value             : ₹{np.mean(baskets):,.2f}")
            print(f"Median customer basket value              : ₹{np.median(baskets):,.2f}")

        elif choice == "5":
            export_all_reports(df)

        elif choice == "6":
            print("Exiting Analyzer.")
            break

        else:
            print("Invalid input. Please enter a number between 1 and 6.")

# Start the interactive menu
run_kirana_analyzer()


  KIRANA STORE BILLING & INVENTORY ANALYZER (1L ROWS)
1. View Category-Wise Revenue & Profit Breakdown
2. View Top 10 Highest-Earning Products
3. Check Low-Stock Inventory Alerts (Stock < 10)
4. View Bulk Discount & Basket Value Statistics
5. Export All Processed Reports to CSV
6. Exit

Enter your choice (1-6): 4

--- Discount & Basket Analytics ---
Line items eligible for 10% bulk discount : 25,584 / 100,000
Total discount amount given               : ₹3,969,505.60
Average customer basket value             : ₹2,712.51
Median customer basket value              : ₹2,444.50

  KIRANA STORE BILLING & INVENTORY ANALYZER (1L ROWS)
1. View Category-Wise Revenue & Profit Breakdown
2. View Top 10 Highest-Earning Products
3. Check Low-Stock Inventory Alerts (Stock < 10)
4. View Bulk Discount & Basket Value Statistics
5. Export All Processed Reports to CSV
6. Exit

Enter your choice (1-6): 1

--- Category Performance ---
     Category  Total_Units_Sold  Gross_Revenue  Discounts_Given  Net_Reven